<img align="left" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="right">Train ML models</h1>
<h3 align="right"><a href="https://colab.research.google.com/github/ocean-data-factory-sweden/kso/blob/main/notebooks/analyse/Train_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a></h3>
<h3 align="right">Written by the KSO Team</h3>

This notebook takes you through the process of importing a baseline model, training it on a dataset and evaluating the quality of the model. If you do not have a project with us yet, you can run the template project to get a taste of how it all works. This notebook assumes that the user has prepared the dataset for model training, see Tutorial #8 for details on the required setup.

🔴 <span style="color:red">&nbsp;NOTE: In order to run this notebook, you need to have a Weights and Biases account. If you want to become a member of our Koster team on Weights and Biases, you may request this access by contacting jurie.germishuys@combine.se. But this is not necessary to run the template project. </span>

# Set up KSO requirements

### (Optional) Enable a development version of KSO

In [ ]:
import os
import sys
from pathlib import Path

KSO_DEV_PATH = Path.cwd().parent.parent
print(KSO_DEV_PATH)
sys.path.insert(0, str(KSO_DEV_PATH))

### Load KSO modules

In [ ]:
%matplotlib inline
import os
import sys
from pathlib import Path

import pandas as pd
import wandb

import kso_utils
print(f"Using kso_utils from {kso_utils.__file__}")

import kso_utils.widgets as kso_widgets
import kso_utils.project_utils as p_utils
import kso_utils.server_utils as s_utils
import kso_utils.yolo_utils as y_utils
from kso_utils.project import ProjectProcessor, MLProjectProcessor
from ipyfilechooser import FileChooser
from IPython.display import display

### Login to wandb

In [ ]:
wandb.login()
api = wandb.Api()
api.runs(path="koster/model-registry")

### Choose whether to use interactive widgets or not

In [ ]:
# USE_WIDGETS = True
USE_WIDGETS = False

### Choose your project

In [ ]:
if USE_WIDGETS:
    project_name_widget = kso_widgets.choose_project()
else:
    projects_csv_path = Path("../../kso_utils/db_starter/projects_list.csv")
    projects_df = pd.read_csv(projects_csv_path)
    print(projects_df["Project_name"])
    project_name = "Template project"

### Initiate project's database

In [ ]:
# Resolve widget value
try:
    project_name = project_name_widget.value
except:
    pass
print(project_name)

# Find project
project = p_utils.find_project(project_name=project_name)
# Initialise pp
mlp = MLProjectProcessor(project)

In [ ]:
# Only for Template Project (downloading prepared data)
s_utils.get_ml_data(project)

# Train the model

### Configure data paths

If you are running the Template project, the output_folder that you want to select is the ml-template-data. The path to this folder is printed in the cell above. For any other project, it is the folder where you have saved your data.

In [ ]:
# Specify path containing the images and labels folders.
if USE_WIDGETS:
    mlp.output_path = kso_widgets.choose_folder(
        project.photo_folder if not project.photo_folder == "None" else ".", "output"
    )
else:
    mlp.output_path = 'ml-template-data'

🔴 <span style="color:red">&nbsp;NOTE: Each model type requires a specific folder structure to be in place. To be able to train your own Object Detection models, your data_path must contain a yml file for data and hyperparameters. See https://github.com/ultralytics/yolov5/wiki/Train-Custom-Data#11-create-datasetyaml. For image classification models, there should be 3 folders (train, val, test) each containing images in class_name folders. For segmentation models, polygon coordinates are also required. </span>

In [ ]:
# Fix important paths
mlp.setup_paths()

### Choose a suitable experiment name

In [ ]:
if USE_WIDGETS:
    exp_name_widget = kso_widgets.choose_experiment_name()
else:
    exp_name = "running_on_lumi"

### Choose model to use for training

In the next cell you will specify the folder (can be any folder of choice) where you want to download the baseline model to, which you will select in the cell after. This baseline model will be used as the starting point for the training.

In [ ]:
# Specify path to download baseline model
if USE_WIDGETS:
    download_folder_widget = kso_widgets.choose_folder(
        project.photo_folder if not project.photo_folder == "None" else ".",
        "model download",
    )
else:
    download_folder = "."

In [ ]:
# Resolve widget value
try:
    download_folder = download_folder_widget.selected
except:
    pass
print(download_folder)

if False: # USE_WIDGETS: # XXX This widget is broken
    weights_widget = mlp.choose_baseline_model(download_folder)
else:
    api = wandb.Api()
    collections = [
        coll 
        for coll in api.artifact_type(
            type_name="model", project="koster/model-registry"
        ).collections()
    ]
    
    model_dict = {}
    for artifact in collections:
        name = artifact.name
        model_dict[name] = artifact
        print(f"{artifact.name:40}: {artifact}")

    # Choose model
    name = 'yolov8m-base-model'
    assert name in model_dict

    # Download model
    for af in model_dict[name].artifacts():
        af.download(download_folder)
    artifact_path = Path(download_folder) / name.replace('-base-model', '.pt')

### Train model with given configuration

The cell below will ask you which batch size and how many epochs you want to use during training. There are no strict rules for this and the best settings will depend on the choice of GPU and some randomness that we have encountered while training models. Therefore it will be some trial and error. As a starting point we advice to use a batch size of 8. For smaller datasets, we have experienced that 50-100 epochs has been sufficient to get good performance on the model (metrics that have reached a plateau), but to not overfit to the training set.

In [ ]:
if USE_WIDGETS:
    batch_size_widget, epochs_widget, img_h_widget, img_w_widget = mlp.choose_train_params()
else:
    batch_size = 1
    epochs = 1
    img_h = 128
    img_w = img_h

In [ ]:
# Give your WandB username, or team name where you want to sent the runs to.
# If you are part of the koster project, you can keep the default 'koster'.
entity = mlp.choose_entity(alt_name=False)

In [ ]:
# Resolve widget values widget by widget
try:
    exp_name = exp_name_widget.value
except:
    pass
print(exp_name)
try:
    artifact_path = weights_widget.artifact_path
except:
    pass
print(artifact_path)
try:
    batch_size = batch_size_widget.value
    epochs = epochs_widget.value
    img_h = img_h_widget.value
    img_w = img_w_widget.value
except:
    pass
print(batch_size, epochs, img_h, img_w)

# XXX This trains the model, uploads data to wandb, and does some logging to mlflow. It'd be great to separate these steps if possible.
# XXX This ignores img_w value
mlp.train_yolo(
    exp_name=exp_name,
    weights=str(artifact_path),
    project=mlp.project_name,
    epochs=epochs,
    batch_size=batch_size,
    img_size=img_h,  # this requires an int
)

# Evaluate model performance

The model is now done with training. To see the loss, precision, recall and some other parameters per training epoch, click on the link in the previous cell. Here you can see your run in Weights and Biases. To evaluate the resulting model, please run the cells below. These execute the standard evaluation process from YOLO.

For a biological evaluation of the model, please see Notebook 6.

In [ ]:
if USE_WIDGETS:
    conf_thres_widget = kso_widgets.choose_eval_params()
else:
    conf_thres = 0.5

In [ ]:
# Choose model: The folder you want to select for eval_model is the folder with your experiment_name.
# XXX This is not used in the cells below
# eval_model = FileChooser(".")
# display(eval_model)

When you run the cell below, you will get some numbers logged on the screen, and 3 files that are stored in the folder 'your_experiment_name'_val.

The numbers logged on the screen represent the following:
* The first 7 numbers are the: mean precision, mean recal, mean average precision calculated at IOU threshold 0.5 (map@0.5), the mean average precision calculated at different IOU thresholds of 0.5-0.95 with steps of 0.05 (map@0.5:0.95) and then 3 training losses based on predicting the box, object or class.
* The array gives the ap@0.5 per class.
* The last 3 numbers are the same as the numbers that are already printed in a line above, where it says 'Speed: … ms per....'

In [ ]:
# Resolve widget value
try:
    conf_thres = conf_thres_widget.value
except:
    pass
print(conf_thres)

# Evaluate YOLO Model on Unseen Test data
mlp.eval_yolo(exp_name=exp_name, conf_thres=conf_thres)

# (Optional) : Enhance annotations using trained model

Enhancement uses the trained model to increase the amount of annotations in the training data. This should only be done in cases where it is absolutely necessary as bad predictions lead to worse predictions when used to train the next iteration of the model.


🔴 <span style="color:red">&nbsp;NOTE: We recommend using a relatively high confidence threshold when enhancing trained models as low confidence predictions could significantly impact the quality of your annotated data. This is currently only available for object detection models.  </span>

In [ ]:
if USE_WIDGETS:
    eh_conf_thres_widget = kso_widgets.choose_eval_params()
else:
    eh_conf_thres = 0.5

In [ ]:
# Choose an input path
# input_path = FileChooser(mlp.project_name)
# display(input_path)
input_path = mlp.output_path  # XXX Is this correct here?

In [ ]:
# Find the project path
# project_path = FileChooser(mlp.project_name)
# display(project_path)
project_path = download_folder  # XXX Is this correct here?

In [ ]:
# Resolve widget value
try:
    eh_conf_thres = eh_conf_thres_widget.value
except:
    pass
print(eh_conf_thres)

mlp.enhance_yolo(
    in_path=input_path,
    project_path=project_path,
    conf_thres=eh_conf_thres,
    img_size=[640, 640],
)

### Choose run to use as enhanced annotations

In [ ]:
# XXX What path should be here?
runs = FileChooser(".")
display(runs)

In [ ]:
# Move enhanced annotations to original run folder (NB: This will replace the original annotations)
mlp.enhance_replace(runs)

#### Once you have moved the new labels to the original label location, you can return to Step 2 and train your model again.

In [ ]:
# END